# RoBERTa-base Fine-tune — LLM Authorship Attribution
**The reliable transformer.** RoBERTa fine-tunes cleanly in fp16 on T4 — no NaN issues.
Expected: **85-90% val accuracy** in ~2.5 hours.

## Why RoBERTa instead of DeBERTa-v3?
DeBERTa-v3 NaN'd across 3 separate configs on free T4 hardware:
- fp16 → NaN (disentangled-attention softmax overflow)
- DataParallel → NaN (forward-pass gather corruption)
- gradient_checkpointing → NaN (recomputation instability)

RoBERTa-base is numerically stable, fp16-safe, ~2x faster, and gives comparable accuracy.
This is the pragmatic engineering choice for a reproducible result.

## Setup:
1. Accelerator → **GPU T4 x2** (we force single GPU in code)
2. Add dataset → attach your RAID parquet dataset
3. Update `DATA_DIR` in Cell 2 if needed
4. **Run All** → ~2.5 hours

In [ ]:
# CELL 1: Install and Imports
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # single GPU = simplest, stable

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'datasets', 'accelerate'], check=True)

import gc, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_cosine_schedule_with_warmup)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, f1_score
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WORKING = '/kaggle/working'
print(f'Device: {DEVICE}  |  GPUs visible: {torch.cuda.device_count()} (forced single)')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
torch.manual_seed(42); np.random.seed(42)
print('Imports OK')

In [ ]:
# CELL 2: Config
CLASSES = ['chatgpt','cohere','cohere-chat','gpt2','gpt3','gpt4',
           'human','llama-chat','mistral','mistral-chat','mpt','mpt-chat']
NUM_CLASSES = 12

# Update DATA_DIR to match your dataset attachment path
DATA_DIR   = '/kaggle/input/llm-12-class'
TRAIN_PATH = f'{DATA_DIR}/train/train.parquet'
VAL_PATH   = f'{DATA_DIR}/val/val.parquet'
TEST_PATH  = f'{DATA_DIR}/test/test.parquet'

CONFIG = {
    'model_name'     : 'roberta-base',
    'max_length'     : 256,
    'batch_size'     : 32,     # RoBERTa fp16 fits comfortably on T4
    'grad_accum'     : 1,
    'num_epochs'     : 3,
    'lr'             : 2e-5,   # RoBERTa standard LR
    'warmup_ratio'   : 0.06,
    'weight_decay'   : 0.01,
    'label_smoothing': 0.0,
    'max_grad_norm'  : 1.0,
}

# fp16 is SAFE and FAST for RoBERTa on T4 (unlike DeBERTa-v3)
USE_FP16 = torch.cuda.is_available()
print(f'Model     : {CONFIG["model_name"]}')
print(f'Precision : {"fp16 (safe for RoBERTa)" if USE_FP16 else "fp32"}')
print('Config ready')

In [ ]:
# CELL 3: Load Data
print('Loading dataset...')

def load_split(path, subset=None):
    df = pd.read_parquet(path)
    if 'generated_by' in df.columns:
        df = df.rename(columns={'generated_by': 'model', 'text': 'generation'})
    df = df[df['model'].isin(CLASSES)].reset_index(drop=True)
    # Drop empty / whitespace-only texts (safety)
    df = df[df['generation'].astype(str).str.strip().str.len() > 0].reset_index(drop=True)
    if subset:
        df = df.groupby('model', group_keys=False).apply(
            lambda g: g.sample(min(len(g), subset), random_state=42)
        ).reset_index(drop=True)
    return df

train_df = load_split(TRAIN_PATH)
val_df   = load_split(VAL_PATH)
test_df  = load_split(TEST_PATH)
print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

le = LabelEncoder().fit(CLASSES)
X_train, y_train = train_df['generation'].values, le.transform(train_df['model'].values)
X_val,   y_val   = val_df['generation'].values,   le.transform(val_df['model'].values)
X_test,  y_test  = test_df['generation'].values,  le.transform(test_df['model'].values)
print('Data ready.')

In [ ]:
# CELL 4: Dataset + Helpers
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = list(texts); self.labels = list(labels)
        self.tok = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(str(self.texts[i]), max_length=self.max_len,
                       padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'label': torch.tensor(int(self.labels[i]), dtype=torch.long)}

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, trues, loss_sum = [], [], 0.0
    crit = nn.CrossEntropyLoss()
    for b in loader:
        out = model(input_ids=b['input_ids'].to(DEVICE),
                    attention_mask=b['attention_mask'].to(DEVICE))
        logits = out.logits.float()
        loss_sum += crit(logits, b['label'].to(DEVICE)).item()
        preds.extend(logits.argmax(1).cpu().numpy())
        trues.extend(b['label'].numpy())
    return accuracy_score(trues, preds), loss_sum/len(loader), np.array(preds), np.array(trues)

print('Helpers defined.')

In [ ]:
# CELL 5: RoBERTa Training (fp16, single GPU, stable)
print('='*55)
print('RoBERTa-base FINE-TUNE')
print('='*55)

tok = AutoTokenizer.from_pretrained(CONFIG['model_name'])
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['model_name'], num_labels=NUM_CLASSES).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters())/1e6:.0f}M')

tr_dl = DataLoader(TextDataset(X_train, y_train, tok, CONFIG['max_length']),
                   batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
vl_dl = DataLoader(TextDataset(X_val, y_val, tok, CONFIG['max_length']),
                   batch_size=CONFIG['batch_size']*2, shuffle=False, num_workers=2)
ts_dl = DataLoader(TextDataset(X_test, y_test, tok, CONFIG['max_length']),
                   batch_size=CONFIG['batch_size']*2, shuffle=False, num_workers=2)

# Optimizer with weight decay separation
no_decay = ['bias', 'LayerNorm.weight']
param_groups = [
    {'params': [p for n,p in model.named_parameters() if not any(nd in n for nd in no_decay)],
     'weight_decay': CONFIG['weight_decay']},
    {'params': [p for n,p in model.named_parameters() if any(nd in n for nd in no_decay)],
     'weight_decay': 0.0},
]
opt = torch.optim.AdamW(param_groups, lr=CONFIG['lr'], eps=1e-8)
n_steps = (len(tr_dl) // CONFIG['grad_accum']) * CONFIG['num_epochs']
warm = int(n_steps * CONFIG['warmup_ratio'])
sch = get_cosine_schedule_with_warmup(opt, warm, n_steps)
crit = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])
scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)

print(f'Steps={n_steps}  Warmup={warm}  lr={CONFIG["lr"]}  fp16={USE_FP16}')
print('Expected: val_acc > 8.3% ep1, ~85-90% ep3  (~2.5h)')

best_acc, best_state = 0.0, None
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
t_start = time.time()

for epoch in range(1, CONFIG['num_epochs'] + 1):
    model.train(); opt.zero_grad()
    running, counted = 0.0, 0
    for step, b in enumerate(tr_dl):
        ids  = b['input_ids'].to(DEVICE)
        mask = b['attention_mask'].to(DEVICE)
        labs = b['label'].to(DEVICE)
        with torch.cuda.amp.autocast(enabled=USE_FP16):
            out  = model(input_ids=ids, attention_mask=mask)
            loss = crit(out.logits.float(), labs) / CONFIG['grad_accum']
        scaler.scale(loss).backward()
        running += loss.item() * CONFIG['grad_accum']; counted += 1
        if (step + 1) % CONFIG['grad_accum'] == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['max_grad_norm'])
            scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad()
        if step % max(1, len(tr_dl) // 10) == 0:
            print(f'  E{epoch} [{step+1:5d}/{len(tr_dl)}]  loss={running/max(counted,1):.4f}  '
                  f'lr={sch.get_last_lr()[0]:.2e}', end='\r')

    val_acc, val_loss, _, _ = evaluate(model, vl_dl)
    avg_loss = running / max(counted, 1)
    history['train_loss'].append(avg_loss); history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    elapsed = (time.time() - t_start) / 60
    print(f'\nEpoch {epoch}/{CONFIG["num_epochs"]} | train_loss={avg_loss:.4f} | '
          f'val_loss={val_loss:.4f} | val_acc={val_acc*100:.2f}% | elapsed={elapsed:.0f}min')
    if val_acc > best_acc:
        best_acc = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        torch.save(best_state, f'{WORKING}/roberta_best.pt')
        print(f'  New best! Saved roberta_best.pt')

print(f'\nBest val acc: {best_acc*100:.2f}%')

In [ ]:
# CELL 6: Test Evaluation + Save Results
print('Evaluating on test set...')
model.load_state_dict(best_state)
test_acc, _, test_preds, test_trues = evaluate(model, ts_dl)
test_f1 = f1_score(test_trues, test_preds, average='macro')

print(f'\n{"="*55}')
print('RoBERTa-base — FINAL RESULTS')
print(f'{"="*55}')
print(f'Val  Acc: {best_acc*100:.2f}%')
print(f'Test Acc: {test_acc*100:.2f}%')
print(f'Macro F1: {test_f1:.4f}')
print(f'\nClassification Report (test set):')
print(classification_report(le.inverse_transform(test_trues),
                            le.inverse_transform(test_preds), digits=3, zero_division=0))

results = {'RoBERTa-base': {
    'val_acc': round(best_acc*100, 2), 'test_acc': round(test_acc*100, 2),
    'macro_f1': round(test_f1, 4),
    'train_time_min': round((time.time()-t_start)/60, 1), 'config': CONFIG}}
with open(f'{WORKING}/roberta_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nSaved: roberta_results.json + roberta_best.pt')

In [ ]:
# CELL 7: Training Curves + Confusion Matrix
import seaborn as sns
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('RoBERTa-base — Training Curves', fontweight='bold')
axes[0].plot(history['train_loss'], 'o-', color='coral', label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], 's-', color='steelblue', label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot([v*100 for v in history['val_acc']], 's-', color='green', linewidth=2)
axes[1].axhline(y=best_acc*100, color='red', linestyle='--', alpha=0.5, label=f'Best: {best_acc*100:.2f}%')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Acc (%)'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{WORKING}/roberta_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

cm = confusion_matrix(test_trues, test_preds)
cm_norm = cm.astype(float) / (cm.sum(1, keepdims=True) + 1e-10)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax, vmin=0, vmax=1, linewidths=0.4)
ax.set_title(f'RoBERTa Confusion Matrix (Test)  —  Acc: {test_acc*100:.2f}%', fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(f'{WORKING}/roberta_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: roberta_training_curves.png + roberta_confusion_matrix.png')
print('\nDownload all from: Kaggle Output tab')